In [4]:
print("hello world")

hello world


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


In [ ]:
# ==========================================
# 1. CREATE THE SAMPLE DATASET
# ==========================================
sample_reviews = [
    "Absolutely wonderful - silky and sexy and comfortable",
    "The material feels soft and luxurious. Very comfortable to wear.",
    "The quality of this dress is excellent and it feels well made.",
    "The fabric feels cheap and started losing shape after one wash.",
    "I love the style of this blouse. It looks modern and elegant.",
    "The fit was perfect and true to size.",
    "This shirt runs very small, so I recommend ordering one size up.",
    "The jeans fit nicely around the waist and hips.",
    "Very comfortable shoes that I can wear all day.",
    "The sweater is cozy, soft, and extremely comfortable.",
    "Beautiful design and stylish enough for both work and dinner.",
    "The stitching is poor and the quality does not justify the price.",
    "The dress is beautiful but the fit is too tight around the chest.",
    "I received many compliments because the style is unique and fashionable.",
    "The pants are comfortable but slightly too long for me.",
    "Excellent quality fabric and the color looks exactly like the pictures.",
    "The material is thin and feels less durable than expected.",
    "This jacket fits perfectly and has a flattering shape.",
    "The style is classic and easy to match with different outfits.",
    "Very soft fabric and comfortable enough for everyday wear.",
    "The sizing was inaccurate and the dress was much larger than expected.",
    "I am impressed with the quality and careful stitching.",
    "This top looks stylish but feels uncomfortable around the shoulders.",
    "The skirt fits well and moves comfortably.",
    "Poor quality zipper and loose threads after only a few uses.",
    "The design is beautiful and the colors are vibrant.",
    "This cardigan is comfortable, warm, and soft.",
    "The fit is awkward and does not match the size chart.",
    "Great quality for the price and the fabric feels durable.",
    "The style is trendy and perfect for casual weekends.",
    "I love how comfortable these leggings are during workouts.",
    "The blouse fits exactly as expected and looks flattering.",
    "The material feels premium and the construction is excellent.",
    "The style was not what I expected from the photos.",
    "These pants are too loose around the waist.",
    "The dress is comfortable and lightweight for summer.",
    "Amazing quality and beautiful attention to detail.",
    "The jacket is stylish but the sleeves are too short.",
    "Soft, comfortable, and easy to wear all day.",
    "The fabric shrank after washing, which was disappointing.",
    "This is one of the most stylish dresses I have purchased.",
    "The fit around the waist is perfect but the length is too long.",
    "The quality exceeded my expectations and feels very durable.",
    "The shirt is uncomfortable because the fabric is rough.",
    "Beautiful style, elegant design, and a very flattering look.",
    "The size was too small even though I ordered my usual size.",
    "The material is soft and the overall quality is impressive.",
    "Very comfortable dress that is perfect for traveling.",
    "The design is fashionable but the fabric quality is average.",
    "These jeans have a perfect fit and are very flattering.",
    "The sweater feels cheap and started pilling quickly.",
    "I love the style and the unique pattern.",
    "The shoes are extremely comfortable and provide good support.",
    "The dress fits beautifully and feels comfortable.",
    "The quality is poor and the seams started coming apart.",
    "The top is stylish, versatile, and easy to dress up.",
    "The pants fit perfectly and the material feels durable.",
    "Comfortable, soft, and perfect for relaxing at home.",
    "The size is too large and the sleeves are much longer than expected.",
    "Excellent fabric quality and beautiful craftsmanship.",
]

reviews = pd.DataFrame({"Review Text": sample_reviews})
reviews.to_csv("womens_clothing_e-commerce_reviews.csv", index=False)
print("Sample dataset created successfully!")


In [ ]:
# ==========================================
# INSTRUCTION 1: Create and store the embeddings
# ==========================================
# We use a lightweight local model 'all-MiniLM-L6-v2' to create embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed the reviews and convert the resulting numpy array to a list
embeddings_array = model.encode(reviews["Review Text"].tolist())
embeddings = embeddings_array.tolist()
print(f"Stored {len(embeddings)} embeddings successfully.")


In [ ]:
# ==========================================
# INSTRUCTION 2: Dimensionality reduction & visualization
# ==========================================
# Use PCA to reduce dimensions down to 2
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(np.array(embeddings))

# Plotting the 2D visual representation
plt.figure(figsize=(10, 8))
sns.scatterplot(x=embeddings_2d[:, 0], y=embeddings_2d[:, 1], alpha=0.7)
plt.title("2D Projection of E-commerce Clothing Review Embeddings")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


In [ ]:
# ==========================================
# INSTRUCTION 3: Feedback categorization
# ==========================================
# Example demonstration tracking topic associations manually or via keywords
print("\n--- Feedback Categorization Examples ---")
topics = ["quality", "fit", "style", "comfort"]
for t in topics:
    matches = reviews[
        reviews["Review Text"].str.contains(t, case=False)
    ].head(2)
    print(f"\nTopic: {t.upper()}")
    for idx, row in matches.iterrows():
        print(f"- {row['Review Text']}")


In [ ]:
# ==========================================
# INSTRUCTION 4: Similarity search function
# ==========================================
def get_closest_reviews(input_review, all_reviews_df, review_embeddings, k=3):
    # Encode target input review
    input_emb = model.encode([input_review])

    # Compute cosine similarity against all stored embeddings
    similarities = cosine_similarity(input_emb, np.array(review_embeddings))[0]

    # Get indices of the top k highest similarity scores
    top_indices = np.argsort(similarities)[::-1][:k]

    # Return the text entries as a list
    return all_reviews_df.iloc[top_indices]["Review Text"].tolist()


# Apply this function to the first review
target_review = reviews["Review Text"].iloc[0]
most_similar_reviews = get_closest_reviews(target_review, reviews, embeddings)

print(f"\n--- Target Review ---\n{target_review}")
print("\n--- Most Similar Reviews (Stored in 'most_similar_reviews') ---")
for i, rev in enumerate(most_similar_reviews, 1):
    print(f"{i}. {rev}")

In [ ]:
# ==========================================
# 1. INITIALIZATION
# ==========================================

client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
# ==========================================
# 2. LOAD DATA
# select only Review Text and This removes missing values (NaN). 
# next arange the serial number.
# ==========================================

reviews = pd.read_csv(
    "womens_clothing_e-commerce_reviews.csv"
)

review_texts = (
    reviews["Review Text"]
    .dropna()
    .reset_index(drop=True)
)

print(f"Number of reviews: {len(review_texts)}")



In [ ]:
# ==========================================
# 3. CREATE EMBEDDINGS
# ==========================================

response = client.embeddings.create(
    input=review_texts.tolist(),
    model=EMBEDDING_MODEL
)

embeddings = [
    item.embedding
    for item in response.data
]

print(f"Embeddings created: {len(embeddings)}")


In [ ]:
# ==========================================
# 4. DIMENSIONALITY REDUCTION WITH t-SNE
# ==========================================

embeddings_array = np.array(embeddings)

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=10
)

embeddings_2d = tsne.fit_transform(
    embeddings_array
)

print(
    f"2D embedding shape: {embeddings_2d.shape}"
)


In [ ]:
# ==========================================
# 5. VISUALIZE EMBEDDINGS
# ==========================================

plt.figure(figsize=(12, 8))

plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    alpha=0.7
)

plt.title(
    "2D Visualization of Review Embeddings"
)

plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")

plt.show()


In [ ]:
# ==========================================
# 6. FEEDBACK CATEGORIZATION
# ==========================================

categories = [
    "Quality",
    "Fit",
    "Style",
    "Comfort"
]

category_descriptions = [
    "reviews discussing product quality, material quality, durability, stitching, fabric, craftsmanship",
    
    "reviews discussing size, sizing, fit, tightness, looseness, waist, length, measurements",
    
    "reviews discussing fashion style, design, appearance, trendiness, elegance, looks",
    
    "reviews discussing comfort, softness, coziness, ease of wearing, pleasant feeling"
]


# Create embeddings for categories

category_response = client.embeddings.create(
    input=category_descriptions,
    model=EMBEDDING_MODEL
)

category_embeddings = np.array([
    item.embedding
    for item in category_response.data
])


# Categorization function

def categorize_feedback(
    text_embedding,
    category_embeddings,
    categories
):
    
    similarities = cosine_similarity(
        [text_embedding],
        category_embeddings
    )[0]
    
    best_index = np.argmax(similarities)
    
    return categories[best_index]


# Categorize all reviews

feedback_categories = [
    categorize_feedback(
        embedding,
        category_embeddings,
        categories
    )
    for embedding in embeddings
]


# Store results

reviews_clean = pd.DataFrame({
    "Review Text": review_texts,
    "Category": feedback_categories
})


print("\nCategorized Reviews:")
print(reviews_clean.head(10))


In [ ]:
# ==========================================
# 7. SIMILARITY SEARCH
# ==========================================

def find_similar_reviews(
    input_text,
    review_texts,
    embeddings,
    client,
    model,
    n=3
):
    
    # Embed input text
    response = client.embeddings.create(
        input=[input_text],
        model=model
    )
    
    input_embedding = response.data[0].embedding
    
    
    # Calculate cosine similarity
    similarities = cosine_similarity(
        [input_embedding],
        embeddings
    )[0]
    
    
    # Get top N most similar indexes
    top_indices = np.argsort(
        similarities
    )[-n:][::-1]
    
    
    # Return reviews
    return [
        review_texts.iloc[i]
        for i in top_indices
    ]


In [ ]:
# ==========================================
# 8. FIND 3 MOST SIMILAR REVIEWS
# ==========================================

example_review = (
    "Absolutely wonderful - silky and sexy and comfortable"
)

most_similar_reviews = find_similar_reviews(
    input_text=example_review,
    review_texts=review_texts,
    embeddings=embeddings,
    client=client,
    model=EMBEDDING_MODEL,
    n=3
)

print("\nMost Similar Reviews:")

for i, review in enumerate(
    most_similar_reviews,
    start=1
):
    print(f"{i}. {review}")